In [ ]:
import os
import json
from PIL import Image
from tqdm import tqdm
from datasets import load_dataset
import random

# --- 1. 설정 ---
print("설정을 시작합니다...")
dataset_name = "suyc21/VMCBench"
split_name = 'dev'

# VQAv2와 동일한 폴더 구조 생성
base_dir = "VMCBench_as_Official_VQAv2_train_only" # train만 저장하도록 경로명 변경
train_image_dir = os.path.join(base_dir, "train2014")
annotation_dir = os.path.join(base_dir, "vqa")

os.makedirs(train_image_dir, exist_ok=True)
os.makedirs(annotation_dir, exist_ok=True)

# --- 2. 데이터셋 로드 및 필터링 ---
print("VMCBench 데이터셋을 로드하고 필터링합니다...")
original_dataset = load_dataset(dataset_name, split=split_name)
# 'image' 필드가 None이 아닌 샘플만 필터링
image_dataset = original_dataset.filter(lambda x: x.get("image") is not None)

print(f"총 유효 샘플 수: {len(image_dataset)}")

# --- 3. 데이터 변환 및 저장 ---
def process_and_save_split(dataset_split, split_key, image_save_dir):
    """지정된 데이터 스플릿을 VQAv2 형식으로 변환하고 저장하는 함수"""
    print(f"'{split_key}' 스플릿을 처리 중입니다...")
    
    questions_list = []
    annotations_list = []
    
    # 'A', 'B', 'C', 'D'를 0, 1, 2, 3으로 매핑
    choice_char_to_index = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

    for example in tqdm(dataset_split, desc=f"Processing {split_key}"):
        # --- 이미지 처리 ---
        image_id = example['index']
        # VQAv2 이미지 파일명 형식: COCO_{split}_{12자리 숫자}.jpg
        image_filename = f"COCO_{split_key}_{str(image_id).zfill(12)}.jpg"
        image_path = os.path.join(image_save_dir, image_filename)
        example['image'].save(image_path)
        
        # --- 질문(Question) JSON 데이터 생성 ---
        # 중요: 질문 텍스트에 보기를 포함시켜 모델이 4지선다형임을 알도록 함
        question_text_base = example['question']
        choices_text = [
            f"A. {example['A']}",
            f"B. {example['B']}",
            f"C. {example['C']}",
            f"D. {example['D']}"
        ]
        
        joined_choices = '\n'.join(choices_text)
        full_question_text = f"{question_text_base}\n{joined_choices}"
        
        question_data = {
            "image_id": image_id,
            "question": full_question_text,
            "question_id": image_id 
        }
        questions_list.append(question_data)

        # --- 정답(Annotation) JSON 데이터 생성 ---
        answer_char = example['answer'] # 'A', 'B', 'C', 'D'
        correct_answer_text = example[answer_char]
        
        # 중요: 'answer_label' 필드 추가 (0, 1, 2, 3)
        answer_numeric_label = choice_char_to_index.get(answer_char)
        if answer_numeric_label is None:
            print(f"경고: 알 수 없는 정답 문자 '{answer_char}' (image_id: {image_id}). 이 샘플을 건너뜨니다.")
            continue # 유효하지 않은 샘플 건너뛰기

        # VQAv2 Annotation 형식에 맞춤
        annotation_data = {
            "question_type": "multiple choice",
            "multiple_choice_answer": correct_answer_text,
            "answers": [ # VQAv2는 여러 답변을 가질 수 있으므로, 10개로 복제
                {
                    "answer": correct_answer_text,
                    "answer_confidence": "yes",
                    "question_id": image_id,
                    "image_id": image_id
                }
            ] * 10, 
            "image_id": image_id,
            "answer_type": "other",
            "question_id": image_id,
            "answer_label": answer_numeric_label # 중요: 0, 1, 2, 3 형태의 레이블 추가
        }
        annotations_list.append(annotation_data)
            
    # --- JSON 파일 저장 ---
    # VQAv2 Question 파일 형식
    final_questions_json = {
        "info": {"description": "VMCBench converted to VQAv2 Questions"},
        "task_type": "Open-Ended",
        "data_type": f"mscoco_{split_key}",
        "questions": questions_list
    }
    question_json_path = os.path.join(annotation_dir, f"v2_OpenEnded_mscoco_{split_key}_questions.json")
    with open(question_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_questions_json, f, indent=4, ensure_ascii=False)

    # VQAv2 Annotation 파일 형식
    final_annotations_json = {
        "info": {"description": "VMCBench converted to VQAv2 Annotations"},
        "data_type": f"mscoco_{split_key}",
        "annotations": annotations_list
    }
    annotation_json_path = os.path.join(annotation_dir, f"v2_mscoco_{split_key}_annotations.json")
    with open(annotation_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_annotations_json, f, indent=4, ensure_ascii=False)

# 전체 데이터셋을 'train2014' 스플릿으로 처리
process_and_save_split(image_dataset, "train2014", train_image_dir)

print("\n--- ✅ 데이터 변환 및 저장 완료 ---")
print("생성된 파일 구조:")
print(f"/{base_dir}")
print(f"├── /train2014/ ({len(os.listdir(train_image_dir))}개 이미지)")
print(f"└── /vqa/")
for fname in sorted(os.listdir(annotation_dir)):
    print(f"    ├── {fname}")

설정을 시작합니다...
VMCBench 데이터셋을 로드하고 필터링합니다...
총 유효 샘플 수: 1000
'train2014' 스플릿을 처리 중입니다...


Processing train2014: 100%|██████████| 1000/1000 [00:12<00:00, 80.00it/s]



--- ✅ 데이터 변환 및 저장 완료 ---
생성된 파일 구조:
/VMCBench_as_Official_VQAv2_train_only
├── /train2014/ (1000개 이미지)
└── /vqa/
    ├── v2_OpenEnded_mscoco_train2014_questions.json
    ├── v2_mscoco_train2014_annotations.json


: 